<a href="https://colab.research.google.com/github/NamishBansal15/substation-detection/blob/main/inference-mapping/naip_image_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# STEP 1 — SETUP
# =========================
from google.colab import drive
drive.mount('/content/drive')

import ee
import pandas as pd
import requests
import os
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

ee.Authenticate()
ee.Initialize(project="INSERT EE PROJECT")


# =========================
# STEP 2 — PATHS
# =========================
BASE_DIR = "/content/drive/MyDrive/dataset-inference/"

DATA_PATH = BASE_DIR + "data/SubsData.parquet"
OUTPUT_DIR = BASE_DIR + "images/naip_images/"
METADATA_PATH = BASE_DIR + "data/image_metadata.csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# =========================
# STEP 3 — LOAD DATA
# =========================
df = pd.read_parquet(DATA_PATH)

# auto-detect columns
possible_lat = ["lat", "latitude", "Latitude", "LAT", "y"]
possible_lon = ["lon", "longitude", "Longitude", "LON", "lng", "x"]

lat_col = next((c for c in possible_lat if c in df.columns), None)
lon_col = next((c for c in possible_lon if c in df.columns), None)

if lat_col is None or lon_col is None:
    raise ValueError(f"Couldn't find lat/lon columns: {df.columns}")

df = df.rename(columns={lat_col: "latitude", lon_col: "longitude"})
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
df = df.dropna(subset=["latitude", "longitude"])

# optional: deduplicate
df = df.drop_duplicates(subset=["latitude", "longitude"])

print(f"Total points: {len(df)}")


# =========================
# STEP 4 — FAST IMAGE FETCH
# =========================
def get_naip_image_fast(lat, lon, sizes=[2048, 1024]):
    try:
        point = ee.Geometry.Point([lon, lat])

        collection = (
            ee.ImageCollection("USDA/NAIP/DOQQ")
            .filterBounds(point)
            .sort("system:time_start", False)
        )

        image = collection.first()
        if image is None:
            return None, None

        image = image.select(['R', 'G', 'B'])
        region = point.buffer(75).bounds()

        for size in sizes:
            try:
                url = image.getThumbURL({
                    "region": region,
                    "dimensions": size,
                    "format": "png"
                })

                response = requests.get(url, timeout=10)

                if response.status_code == 200:
                    return response.content, size

            except:
                continue

        return None, None

    except:
        return None, None


# =========================
# STEP 5 — PARALLEL WORKER
# =========================
def process_row(idx, row):
    lat = row["latitude"]
    lon = row["longitude"]

    for attempt in range(3):  # retry logic
        img_data, size = get_naip_image_fast(lat, lon)

        if img_data is not None:
            filename = f"substation_{idx}.png"
            filepath = os.path.join(OUTPUT_DIR, filename)

            with open(filepath, "wb") as f:
                f.write(img_data)

            return {
                "id": idx,
                "latitude": lat,
                "longitude": lon,
                "image_path": filepath,
                "resolution": size
            }

        time.sleep(1)  # small backoff

    return None


# =========================
# STEP 6 — PARALLEL EXECUTION
# =========================
MAX_WORKERS = 12   # 🔥 adjust (8–16 is ideal for Colab)

metadata = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(process_row, idx, row): idx
        for idx, row in df.iterrows()
    }

    for future in tqdm(as_completed(futures), total=len(futures)):
        result = future.result()
        if result is not None:
            metadata.append(result)


# =========================
# STEP 7 — SAVE METADATA
# =========================
metadata_df = pd.DataFrame(metadata)
metadata_df.to_csv(METADATA_PATH, index=False)

print(f"✅ Saved {len(metadata_df)} images!")

Mounted at /content/drive
Total points: 11085


100%|██████████| 11085/11085 [28:14<00:00,  6.54it/s]

✅ Saved 11085 images!
